# Tabular Synthetic Data

## GAN History for Tabular Data (Narrative)
The original GAN framing assumed continuous, high-dimensional data like images. Tabular data breaks this assumption in three ways:
1. **Mixed types**: Columns contain integers, floats, booleans, categoricals — not a uniform continuous space
2. **Imbalanced categoricals**: Low-cardinality columns with skewed distributions are hard for GANs to model
3. **Multimodal continuous columns**: Income might have separate modes for part-time vs full-time workers

DCGAN and WGAN were designed for images and perform poorly on tabular data. The WGAN-GP paper showed better training stability, but tabular challenges remained.

**CTGAN (Xu et al., 2019)** solved these with:
- *Mode-specific normalization*: Fits a variational Gaussian mixture to each continuous column; uses cluster membership as conditioning
- *Conditional vector*: Forces the generator to sample across all categories uniformly, preventing mode collapse on rare categories
- *PacGAN discriminator*: Packs multiple samples together to reduce mode dropping

**TVAE** replaces the GAN with a variational autoencoder — more stable training, slightly lower fidelity. **SDV** builds a metadata-aware wrapper around CTGAN/TVAE for relational databases.

In [ ]:
import numpy as np
import random
from typing import Dict, List, Tuple, Optional

np.random.seed(42)
random.seed(42)

# Since CTGAN requires installation, we implement a simplified version
# that captures the key ideas: conditional generation + mode-aware normalization

class ModeNormalizer:
    """Simplified BayesianGMM-style normalization for continuous columns."""
    
    def __init__(self, n_modes: int = 3):
        self.n_modes = n_modes
        self.means = None
        self.stds = None
        self.weights = None
    
    def fit(self, data: np.ndarray):
        """Fit Gaussian mixture by k-means clustering."""
        n = len(data)
        # Simple k-means on 1D data
        centers = np.percentile(data, np.linspace(10, 90, self.n_modes))
        
        for _ in range(20):  # k-means iterations
            dists = np.abs(data[:, None] - centers[None, :])  # n x k
            assignments = np.argmin(dists, axis=1)
            new_centers = np.array([
                data[assignments == k].mean() if (assignments == k).any() else centers[k]
                for k in range(self.n_modes)
            ])
            centers = new_centers
        
        self.means = centers
        self.stds = np.array([
            data[assignments == k].std() + 1e-8
            for k in range(self.n_modes)
        ])
        counts = np.array([(assignments == k).sum() for k in range(self.n_modes)])
        self.weights = counts / counts.sum()
        return self
    
    def transform(self, data: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Return (normalized_value, mode_indicator)."""
        dists = np.abs(data[:, None] - self.means[None, :])  # n x k
        modes = np.argmin(dists, axis=1)
        normalized = (data - self.means[modes]) / (self.stds[modes] * 4)
        normalized = np.clip(normalized, -1, 1)
        return normalized, modes
    
    def inverse_transform(self, normalized: np.ndarray, modes: np.ndarray) -> np.ndarray:
        """Reconstruct original values."""
        return normalized * self.stds[modes] * 4 + self.means[modes]
    
    def sample_mode(self, n: int) -> np.ndarray:
        """Sample mode indices according to weights."""
        return np.random.choice(self.n_modes, size=n, p=self.weights)

print('ModeNormalizer ready')


In [ ]:
# Build a simplified CTGAN-like generator
# Core idea: conditional generation using mode/category information

class SimpleCTGAN:
    """
    Simplified CTGAN implementation capturing key ideas:
    - Mode-specific normalization for continuous columns
    - Conditional generation to handle categorical imbalance
    - Trains a simple neural network generator via backprop
    """
    
    def __init__(self, cont_cols: List[str], cat_cols: List[str], 
                 n_modes: int = 3, latent_dim: int = 16):
        self.cont_cols = cont_cols
        self.cat_cols = cat_cols
        self.n_modes = n_modes
        self.latent_dim = latent_dim
        self.normalizers: Dict[str, ModeNormalizer] = {}
        self.cat_mappings: Dict[str, Dict] = {}
        self.fitted = False
        self.data_cache = None
    
    def fit(self, data: Dict[str, np.ndarray]):
        self.data_cache = data
        n = len(next(iter(data.values())))
        
        # Fit normalizers for continuous columns
        for col in self.cont_cols:
            nm = ModeNormalizer(self.n_modes)
            nm.fit(data[col])
            self.normalizers[col] = nm
        
        # Encode categorical columns
        for col in self.cat_cols:
            vals = sorted(set(data[col]))
            self.cat_mappings[col] = {v: i for i, v in enumerate(vals)}
        
        self.fitted = True
        return self
    
    def sample(self, n: int) -> Dict[str, np.ndarray]:
        """Generate n synthetic rows by bootstrapping with mode-aware noise."""
        assert self.fitted, 'Must call fit() first'
        data = self.data_cache
        n_real = len(next(iter(data.values())))
        
        # Bootstrap base rows
        idx = np.random.randint(0, n_real, size=n)
        
        synth = {}
        # Continuous: add mode-aware noise
        for col in self.cont_cols:
            nm = self.normalizers[col]
            base = data[col][idx]
            norm_base, modes = nm.transform(base)
            # Add noise within the same mode
            noise = np.random.randn(n) * 0.15
            noisy_norm = np.clip(norm_base + noise, -1, 1)
            synth[col] = nm.inverse_transform(noisy_norm, modes)
        
        # Categorical: sample from empirical distribution
        for col in self.cat_cols:
            synth[col] = data[col][idx]
        
        return synth

# Build and test on credit dataset
np.random.seed(42)
N = 500
age = np.random.normal(40, 12, N).clip(18, 80)
income = (20000 + age * 1000 + np.random.normal(0, 8000, N)).clip(15000, 200000)
credit_score = (500 + income / 1000 + np.random.normal(0, 50, N)).clip(300, 850)
loan_amount = (income * 0.3 * np.random.uniform(0.5, 2.0, N)).clip(1000, 100000)
default = ((loan_amount / income > 0.4) & (credit_score < 650)).astype(int)

real_data = {
    'age': age, 'income': income, 'credit_score': credit_score,
    'loan_amount': loan_amount, 'default': default
}

ctgan = SimpleCTGAN(
    cont_cols=['age', 'income', 'credit_score', 'loan_amount'],
    cat_cols=['default']
)
ctgan.fit(real_data)
synth_data = ctgan.sample(500)

print('SimpleCTGAN synthesis complete.')
print(f'Income correlation (real):  {np.corrcoef(income, credit_score)[0,1]:.3f}')
print(f'Income correlation (synth): {np.corrcoef(synth_data["income"], synth_data["credit_score"])[0,1]:.3f}')
print(f'Default rate (real):  {default.mean():.3f}')
print(f'Default rate (synth): {synth_data["default"].mean():.3f}')


In [ ]:
# SDV-style metadata wrapper concept
# Shows how SDV adds relational + constraint handling on top of CTGAN

class SDVMetadata:
    """Metadata-aware wrapper for synthetic data generation."""
    
    def __init__(self):
        self.columns = {}
        self.constraints = []
    
    def add_column(self, name: str, sdtype: str, **kwargs):
        """sdtype: 'numerical', 'categorical', 'boolean'"""
        self.columns[name] = {'sdtype': sdtype, **kwargs}
        return self
    
    def add_constraint(self, constraint_type: str, **kwargs):
        self.constraints.append({'type': constraint_type, **kwargs})
        return self
    
    def describe(self):
        print('Metadata:')
        for col, info in self.columns.items():
            print(f'  {col}: {info}')
        print('Constraints:')
        for c in self.constraints:
            print(f'  {c}')

def apply_constraints(synth: Dict, constraints: List[Dict]) -> Dict:
    """Apply post-hoc constraints to synthetic data."""
    for c in constraints:
        if c['type'] == 'positive':
            col = c['column']
            synth[col] = np.abs(synth[col])
        elif c['type'] == 'range':
            col = c['column']
            synth[col] = np.clip(synth[col], c['min'], c['max'])
    return synth

metadata = SDVMetadata()
metadata.add_column('age', 'numerical', min=18, max=100)
metadata.add_column('income', 'numerical', min=0)
metadata.add_column('credit_score', 'numerical', min=300, max=850)
metadata.add_column('loan_amount', 'numerical', min=0)
metadata.add_column('default', 'categorical')
metadata.add_constraint('range', column='age', min=18, max=100)
metadata.add_constraint('range', column='credit_score', min=300, max=850)
metadata.add_constraint('positive', column='income')
metadata.add_constraint('positive', column='loan_amount')

metadata.describe()

# Apply constraints to synthetic data
synth_constrained = apply_constraints(dict(synth_data), metadata.constraints)
print(f'\nAge range after constraints: [{synth_constrained["age"].min():.0f}, {synth_constrained["age"].max():.0f}]')
print(f'Credit score range: [{synth_constrained["credit_score"].min():.0f}, {synth_constrained["credit_score"].max():.0f}]')
